# Inter-rater agreement (Fleiss' kappa)

A stratified random sample of 200 laboratory images (50 per class) is drawn and given to three raters, who label each photograph independently. Fleiss' kappa is computed over the images that all three raters labelled with one of the four class names.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, os, random, shutil
import numpy as np
import pandas as pd
from pathlib import Path

BASE_DIR      = "/content/drive/MyDrive/UMK/Datasets/MCC-02-dataset"
ANN_JSON      = os.path.join(BASE_DIR, "annotations_coco.json")
IMG_DIR       = os.path.join(BASE_DIR, "images")
OUTPUT_DIR    = os.path.join(BASE_DIR, "kappa_reannot")
N_SAMPLE      = 200      # 50 per class
RANDOM_SEED   = 42
CLASS_NAMES   = ['Broken', 'Fermented', 'Moldy', 'Unfermented']

## 1. Draw the sample and create the rater template

In [ ]:
def select_sample_images():
    """Draw a stratified random sample (N_SAMPLE // 4 per class) and write the rater template."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    with open(ANN_JSON) as f:
        coco = json.load(f)

    cat_map = {c['id']: c['name'] for c in coco['categories']}
    img_map = {img['id']: img['file_name'] for img in coco['images']}

    # One entry per image, labelled with its dataset class
    class_images = {cls: [] for cls in CLASS_NAMES}
    seen = set()
    for ann in coco['annotations']:
        img_id  = ann['image_id']
        cls_name = cat_map[ann['category_id']]
        if img_id not in seen:
            seen.add(img_id)
            class_images[cls_name].append({
                'image_id':   img_id,
                'file_name':  img_map[img_id],
                'true_class': cls_name
            })

    n_per_class = N_SAMPLE // len(CLASS_NAMES)
    random.seed(RANDOM_SEED)
    sampled = []
    for cls in CLASS_NAMES:
        pool = class_images[cls]
        n    = min(n_per_class, len(pool))
        sampled.extend(random.sample(pool, n))

    random.shuffle(sampled)   # raters see the images in random order

    # Template given to each rater (no dataset label)
    df_officer = pd.DataFrame([{
        'no':        i + 1,
        'file_name': s['file_name'],
        'label':     ''
    } for i, s in enumerate(sampled)])

    # Dataset labels of the sample, kept separately from the raters
    df_gt = pd.DataFrame([{
        'no':         i + 1,
        'file_name':  s['file_name'],
        'true_class': s['true_class']
    } for i, s in enumerate(sampled)])

    template_path = os.path.join(OUTPUT_DIR, "annotation_template_OFFICER.csv")
    df_officer.to_csv(template_path, index=False)

    gt_path = os.path.join(OUTPUT_DIR, "ground_truth_DO_NOT_SHARE.csv")
    df_gt.to_csv(gt_path, index=False)

    print(f"Sampled {len(sampled)} images")
    for cls in CLASS_NAMES:
        n = sum(1 for s in sampled if s['true_class'] == cls)
        print(f"  {cls:<15}: {n}")
    print(f"Rater template : {template_path}")
    print(f"Dataset labels : {gt_path}")

    # Copy the sampled images to a separate folder for the raters
    sample_img_dir = os.path.join(OUTPUT_DIR, "sample_images")
    os.makedirs(sample_img_dir, exist_ok=True)
    n_copied = 0
    n_missing = 0
    for s in sampled:
        src = os.path.join(IMG_DIR, s['file_name'])
        dst = os.path.join(sample_img_dir, s['file_name'])
        if os.path.exists(src):
            shutil.copy2(src, dst)
            n_copied += 1
        else:
            n_missing += 1
    print(f"Images copied to sample_images/: {n_copied} copied, {n_missing} missing")

    return df_gt

## 2. Compute Fleiss' kappa from the returned rater files

Expects `labels_officer1.csv`, `labels_officer2.csv` and `labels_officer3.csv` in `OUTPUT_DIR`.

In [ ]:
def compute_kappa_from_officer_csvs():
    """Fleiss' kappa over images that all three raters labelled with one of CLASS_NAMES."""
    files = {
        'officer_1': os.path.join(OUTPUT_DIR, "labels_officer1.csv"),
        'officer_2': os.path.join(OUTPUT_DIR, "labels_officer2.csv"),
        'officer_3': os.path.join(OUTPUT_DIR, "labels_officer3.csv"),
    }

    missing = [k for k, v in files.items() if not os.path.exists(v)]
    if missing:
        print(f"Missing rater files: {missing}")
        return None

    # Labels outside CLASS_NAMES are excluded
    officer_data = {}
    for officer, fpath in files.items():
        df = pd.read_csv(fpath)
        df['label'] = df['label'].str.strip()
        valid_mask = df['label'].isin(CLASS_NAMES)
        n_invalid = (~valid_mask).sum()
        if n_invalid > 0:
            print(f"{officer}: {n_invalid} labels outside the four classes excluded")
            df = df[valid_mask]
        officer_data[officer] = dict(zip(df['file_name'], df['label']))
        print(f"{officer}: {len(df)} valid labels")

    # Images with a valid label from all three raters
    common_imgs = set(officer_data['officer_1'].keys())
    for o in ['officer_2', 'officer_3']:
        common_imgs &= set(officer_data[o].keys())
    common_imgs = sorted(common_imgs)
    print(f"Images labelled by all three raters: {len(common_imgs)}")

    cat_to_idx = {c: i for i, c in enumerate(CLASS_NAMES)}
    n_cat      = len(CLASS_NAMES)
    n_raters   = 3
    N          = len(common_imgs)

    rating_matrix  = np.zeros((N, n_cat), dtype=int)
    disagree_cases = []

    for i, fname in enumerate(common_imgs):
        labels = [officer_data[o][fname] for o in files.keys()]
        for lbl in labels:
            if lbl in cat_to_idx:
                rating_matrix[i, cat_to_idx[lbl]] += 1
        if len(set(labels)) > 1:
            disagree_cases.append({'file': fname, 'labels': labels})

    # Fleiss' kappa
    P_i     = np.array([
        (1 / (n_raters * (n_raters - 1))) *
        np.sum(rating_matrix[i] * (rating_matrix[i] - 1))
        for i in range(N)
    ])
    P_bar   = float(np.mean(P_i))
    p_j     = rating_matrix.sum(axis=0) / (N * n_raters)
    P_bar_e = float(np.sum(p_j ** 2))
    kappa   = (P_bar - P_bar_e) / (1 - P_bar_e)

    # Landis & Koch (1977)
    if kappa >= 0.81:
        interp = "Almost perfect (0.81–1.00)"
    elif kappa >= 0.61:
        interp = "Substantial (0.61–0.80)"
    elif kappa >= 0.41:
        interp = "Moderate (0.41–0.60)"
    elif kappa >= 0.21:
        interp = "Fair (0.21–0.40)"
    else:
        interp = "Slight or poor (< 0.21)"

    print(f"N images        : {N}")
    print(f"N raters        : {n_raters}")
    print(f"P_bar (observed): {P_bar:.6f}")
    print(f"P_e (chance)    : {P_bar_e:.6f}")
    print(f"Fleiss' kappa   : {kappa:.4f}")
    print(f"Interpretation  : {interp}")

    print("Share of assignments per class:")
    for c, p in zip(CLASS_NAMES, p_j):
        print(f"  {c:<15}: {p:.4f}")

    print(f"Disagreements: {len(disagree_cases)} / {N} "
          f"({100*len(disagree_cases)/N:.1f}%)")
    if disagree_cases:
        df_dis = pd.DataFrame(disagree_cases)
        dis_path = os.path.join(OUTPUT_DIR, "disagreement_cases.csv")
        df_dis.to_csv(dis_path, index=False)
        print(f"Disagreement cases saved to {dis_path}")
        for d in disagree_cases[:5]:
            print(f"  {d['file']}: {d['labels']}")

    return kappa, disagree_cases

## Run

Run step 1 once to create the sample; run step 2 after the raters return their files.

In [ ]:
# select_sample_images()
compute_kappa_from_officer_csvs()